In [1]:
!pip install lxml

In [ ]:
from lxml import etree
import re

# =========================
# PARAMÈTRES
# =========================
ARK = "bpt6k1051278h"
START_F = 403

BASE_URL = f"https://gallica.bnf.fr/ark:/12148/{ARK}/f{{}}.item"

NSMAP = {
    None: "http://www.tei-c.org/ns/1.0",
    "xi": "http://www.w3.org/2001/XInclude"
}

# =========================
# PARSER (IMPORTANT)
# =========================
parser = etree.XMLParser(
    remove_blank_text=True,
    strip_cdata=False
)

tree = etree.parse("../../corpus/Peinture/Piles_CoursPeinture.xml", parser)
root = tree.getroot()

# =========================
# EXTRACTION fXXX
# =========================
def extract_f_number(facs_value):
    match = re.search(r'/f(\d+)\.item', facs_value)
    if match:
        return int(match.group(1))
    return None

# =========================
# TRAITEMENT DES <pb>
# =========================
current_f = START_F

for pb in root.xpath('//tei:pb', namespaces={'tei': NSMAP[None]}):
    facs = pb.get('facs')

    if facs:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
    else:
        current_f += 1
        pb.set('facs', BASE_URL.format(current_f))

# =========================
# FIX xi:include (éviter ns1)
# =========================
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

# =========================
# ÉCRITURE SANS ESPACES EN TROP
# =========================
tree.write(
    "output.xml",
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print("✔ XML propre, namespaces conservés, pas de ns1")

✔ XML propre, namespaces conservés, pas de ns1


In [16]:
from lxml import etree
import re
from pathlib import Path

# =========================
# PARAMÈTRES
# =========================
ARK = "bpt6k1051278h"
START_F = 4
START_ARCHIVE = 4

DOCUMENT = "Alberti_DellaPittura.xml"
TYPE_TEXTE = "Peinture"
DOCUMENT_PATH = Path("../../corpus") / TYPE_TEXTE / DOCUMENT

BASE_ARCHIVE_URL = "https://iiif.archive.org/image/iiif/3/gri_pitturexxxxx00albe%2Fgri_pitturexxxxx00albe_jp2.zip%2Fgri_pitturexxxxx00albe_jp2%2Fgri_pitturexxxxx00albe_{:04d}.jp2/full/max/0/default.jpg"

NAMESPACES = {'tei': 'http://www.tei-c.org/ns/1.0', 'xi': 'http://www.w3.org/2001/XInclude'}

# =========================
# PARSER & TRAITEMENT
# =========================
def extract_f_number(facs_value):
    if not facs_value: return None
    match = re.search(r'/f(\d+)', facs_value)
    return int(match.group(1)) if match else None

parser = etree.XMLParser(remove_blank_text=True)
tree = etree.parse(str(DOCUMENT_PATH), parser)
root = tree.getroot()

# Initialisation
current_f = START_F
offset = START_ARCHIVE - START_F

# Correction de la boucle : on suit l'ordre du document
for i, pb in enumerate(root.xpath('//tei:pb', namespaces=NAMESPACES)):
    facs = pb.get('facs')
    
    # Si c'est la toute première page, on reste sur START_F
    # Sinon, on analyse pour voir s'il faut sauter à un nouveau numéro ou juste faire +1
    if i > 0:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
        else:
            current_f += 1

    # Calcul de l'index Archive
    archive_idx = current_f + offset
    
    # MISE À JOUR SYSTÉMATIQUE
    pb.set('facs', BASE_ARCHIVE_URL.format(archive_idx))
    
    # Petit debug pour la console
    print(f"Page traitée : Gallica f{current_f} -> Archive _{archive_idx:04d}")

# =========================
# FINALISATION
# =========================
# Nettoyage namespaces XInclude
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

tree.write("output.xml", encoding="UTF-8", xml_declaration=True, pretty_print=True)

Page traitée : Gallica f4 -> Archive _0004
Page traitée : Gallica f5 -> Archive _0005
Page traitée : Gallica f6 -> Archive _0006
Page traitée : Gallica f7 -> Archive _0007
Page traitée : Gallica f8 -> Archive _0008
Page traitée : Gallica f9 -> Archive _0009
Page traitée : Gallica f10 -> Archive _0010
Page traitée : Gallica f11 -> Archive _0011
Page traitée : Gallica f12 -> Archive _0012
Page traitée : Gallica f13 -> Archive _0013
Page traitée : Gallica f14 -> Archive _0014
Page traitée : Gallica f15 -> Archive _0015
Page traitée : Gallica f16 -> Archive _0016
Page traitée : Gallica f17 -> Archive _0017
Page traitée : Gallica f18 -> Archive _0018
Page traitée : Gallica f19 -> Archive _0019
Page traitée : Gallica f20 -> Archive _0020
Page traitée : Gallica f21 -> Archive _0021
Page traitée : Gallica f22 -> Archive _0022
Page traitée : Gallica f23 -> Archive _0023
Page traitée : Gallica f24 -> Archive _0024
Page traitée : Gallica f25 -> Archive _0025
Page traitée : Gallica f26 -> Archive 